<a href="https://colab.research.google.com/github/shreyanshxt/Autism-Detection/blob/hand-update/Hand_writing_update(1.2).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [2]:
import os

LOCAL_DATA = '/content/handwriting'
os.makedirs(LOCAL_DATA, exist_ok=True)

!unzip -q "/content/drive/MyDrive/Handwriting/Handwritting-20260202T093629Z-3-001.zip" -d {LOCAL_DATA}/Handwriting/

print("Extraction complete. Data is ready for preprocessing.")

Extraction complete. Data is ready for preprocessing.


In [ ]:
import os
import torch
import pandas as pd
import torch.nn as nn
from torchvision import models
import torch.optim as optim
import torchvision.transforms as transforms
from torch.utils.data import Dataset, DataLoader
import torchvision.models as models
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import StratifiedKFold
import numpy as np# Define device (GPU if available)


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Path to dataset
BASE_DIR = "/content/handwriting/Handwriting/Handwritting"

# Constants
IMG_SIZE = (224, 224)
BATCH_SIZE = 32
EPOCHS = 30
SEED = 42

def create_metadata_dataframe(base_dir):
    """
    Create a dataframe with metadata for all images in the dataset
    """
    data = []

    # First level: ASD, ASD with CD, Non-ASD
    for condition in os.listdir(base_dir):
        condition_path = os.path.join(base_dir, condition)
        if not os.path.isdir(condition_path):
            continue

        # Second level: Mild, Moderate, Severe
        for severity in os.listdir(condition_path):
            severity_path = os.path.join(condition_path, severity)
            if not os.path.isdir(severity_path):
                continue

            # Third level: Age groups
            for age_group in os.listdir(severity_path):
                age_path = os.path.join(severity_path, age_group)
                if not os.path.isdir(age_path):
                    continue

                # Fourth level: Activity types
                for activity in os.listdir(age_path):
                    activity_path = os.path.join(age_path, activity)
                    if not os.path.isdir(activity_path):
                        continue

                    # Get all image files
                    for img_file in os.listdir(activity_path):
                        if img_file.lower().endswith(('.jpg', '.jpeg', '.png')):
                            img_path = os.path.join(activity_path, img_file)

                            # Add metadata for this image
                            data.append({
                                'filepath': img_path,
                                'condition': condition,
                                'severity': severity,
                                'age_group': age_group,
                                'activity': activity,
                                'filename': img_file
                            })

    # Create DataFrame
    df = pd.DataFrame(data)

    # Extract numeric age range for easier analysis
    df['age_min'] = df['age_group'].str.extract(r'(\d+)').astype(float)

    # Add binary label for autism vs non-autism
    df['has_autism'] = df['condition'].apply(lambda x: 'no' if x == 'Non-ASD' else 'yes')

    # Normalize activity names (fix spelling variations)
    activity_mapping = {
        'Handwriitng': 'Handwriting',
        'Handwritng': 'Handwriting',
        'handwriting': 'Handwriting'
    }
    df['activity'] = df['activity'].replace(activity_mapping)

    return df

# Custom Dataset Class
class ASDDataset(Dataset):
    def __init__(self, dataframe, transform=None):
        self.data = dataframe
        self.transform = transform

        # Encode labels
        self.label_encoder = LabelEncoder()
        self.data['encoded_label'] = self.label_encoder.fit_transform(self.data['has_autism'])

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        img_path = self.data.iloc[idx]['filepath']
        label = self.data.iloc[idx]['encoded_label']

        # Load image
        from PIL import Image
        image = Image.open(img_path).convert('RGB')

        # Apply transformations
        if self.transform:
            image = self.transform(image)

        return image, label

skf = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)
fold_metrics = {
    "accuracy": [],
    "recall": [],
    "f1": [],
    "balanced_acc": [],
    "mcc": [],
    "roc_auc": []
}

# Define Transformations
transform = transforms.Compose([
    transforms.Resize(IMG_SIZE),

    # Handwriting-safe augmentations
    transforms.RandomRotation(degrees=10),
    transforms.RandomAffine(
        degrees=0,
        translate=(0.05, 0.05),
        scale=(0.95, 1.05)
    ),
    transforms.RandomHorizontalFlip(p=0.3),

    transforms.ToTensor(),

    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

# Create Metadata DataFrame
metadata_df = create_metadata_dataframe(BASE_DIR)
for fold, (train_idx, val_idx) in enumerate(
    skf.split(metadata_df, metadata_df["has_autism"])
):
    print(f"\n========== Fold {fold+1}/5 ==========")

# Split the data
train_df, test_df = train_test_split(metadata_df, test_size=0.2, random_state=SEED, stratify=metadata_df['has_autism'])
train_df = metadata_df.iloc[train_idx]
val_df   = metadata_df.iloc[val_idx]
# Create Datasets
train_dataset = ASDDataset(train_df, transform=transform)
test_dataset = ASDDataset(test_df, transform=transform)
val_dataset   = ASDDataset(val_df, transform=transform)
# Create Data Loaders
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)
val_loader   = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)


# Load Pretrained ConvNeXt
model = models.convnext_tiny(weights="IMAGENET1K_V1")

# Freeze all parameters initially
for param in model.parameters():
    param.requires_grad = False

# Unfreeze the parameters of the last stage of the feature extractor (model.features[3])
# The ConvNeXt 'features' module typically has 4 stages, indexed 0 to 3.
for param in model.features[3].parameters():
    param.requires_grad = True

# Get number of classes from the dataset (this will be 2 for binary classification)
original_num_classes = len(train_dataset.label_encoder.classes_)

# Replace classifier head. For binary classification with BCEWithLogitsLoss, output 1 logit.
num_features = model.classifier[2].in_features
model.classifier[2] = nn.Linear(num_features, 1)

# Move model to GPU if available
model = model.to(device);

class FocalLoss(nn.Module):
    def __init__(self, alpha=0.8, gamma=2.0):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.bce = nn.BCEWithLogitsLoss(reduction='none')

    def forward(self, logits, targets):
        # For BCEWithLogitsLoss, targets should have the same shape as logits.
        # If logits is [BATCH_SIZE, 1], targets should be [BATCH_SIZE, 1].
        # Original labels are [BATCH_SIZE], so we unsqueeze them.
        targets = targets.float().unsqueeze(1)

        bce_loss = self.bce(logits, targets)
        probs = torch.sigmoid(logits)

        pt = torch.where(targets == 1, probs, 1 - probs)
        alpha_t = torch.where(targets == 1, self.alpha, 1 - self.alpha)

        loss = alpha_t * (1 - pt) ** self.gamma * bce_loss
        return loss.mean()

# Define Loss and Optimizer
criterion = FocalLoss(alpha=0.75, gamma=2)

# Configure optimizer to train only the unfrozen layers (features[3] and classifier)
optimizer = torch.optim.AdamW([
    {"params": model.features[3].parameters(), "lr": 1e-5},
    {"params": model.classifier.parameters(), "lr": 1e-7}
], weight_decay=1e-4)


# Training Loop
print("Starting Training...")
for epoch in range(EPOCHS):
    model.train()
    running_loss = 0.0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device);

        # Forward Pass
        outputs = model(images)
        loss = criterion(outputs, labels)

        # Backward Pass
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    print(f"Epoch [{epoch+1}/{EPOCHS}], Loss: {running_loss/len(train_loader):.4f}")

# Evaluate Model
model.eval()
correct = 0
total = 0
with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        # For binary output (1 logit), use sigmoid to get probability and then round for prediction
        predicted = (torch.sigmoid(outputs) > 0.5).long().squeeze(1) # Squeeze to match labels shape
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

print(f"Test Accuracy: {100 * correct / total:.2f}%")

# Save Model
torch.save({
    'model_state_dict': model.state_dict(),
    'label_encoder': train_dataset.label_encoder
}, "resnet18_asd_gestures.pth")


metrics = evaluate_model(model, val_loader)

for k in fold_metrics:
        fold_metrics[k].append(metrics[k])


def confusion_counts(y_true, y_pred):
    TP = ((y_true == 1) & (y_pred == 1)).sum()
    TN = ((y_true == 0) & (y_pred == 0)).sum()
    FP = ((y_true == 0) & (y_pred == 1)).sum()
    FN = ((y_true == 1) & (y_pred == 0)).sum()
    return TP, FP, FN, TN
def recall_score(TP, FN, eps=1e-8):
    return TP / (TP + FN + eps)
def precision_score(TP, FP, eps=1e-8):
    return TP / (TP + FP + eps)
def f1_score(precision, recall, eps=1e-8):
    return 2 * precision * recall / (precision + recall + eps)
def balanced_accuracy(TP, FP, FN, TN, eps=1e-8):
    sensitivity = TP / (TP + FN + eps)
    specificity = TN / (TN + FP + eps)
    return 0.5 * (sensitivity + specificity)
import math

def mcc_score(TP, FP, FN, TN, eps=1e-8):
    numerator = TP * TN - FP * FN
    denominator = math.sqrt(
        (TP + FP) * (TP + FN) * (TN + FP) * (TN + FN)
    ) + eps
    return numerator / denominator
model.eval()

all_preds = []
all_labels = []

with torch.no_grad():
    for images, labels in test_loader: # Changed val_loader to test_loader
        images, labels = images.to(device), labels.to(device)

        logits = model(images)
        # For binary output (1 logit), use sigmoid and round for prediction
        preds = (torch.sigmoid(logits) > 0.5).long().squeeze(1)

        all_preds.append(preds.cpu())
        all_labels.append(labels.cpu())

y_pred = torch.cat(all_preds)
y_true = torch.cat(all_labels)

TP, FP, FN, TN = confusion_counts(y_true, y_pred)

recall = recall_score(TP, FN)
precision = precision_score(TP, FP)
f1 = f1_score(precision, recall)
bal_acc = balanced_accuracy(TP, FP, FN, TN)
mcc = mcc_score(TP, FP, FN, TN)

print(f"Recall: {recall:.4f}")
print(f"F1: {f1:.4f}")
print(f"Balanced Acc: {bal_acc:.4f}")
print(f"MCC: {mcc:.4f}")

# Additional Logging
print("\nDataset Information:")
print(f"Total Images: {len(metadata_df)}")
print(f"Training Images: {len(train_df)}")
print(f"Testing Images: {len(test_df)}")
print("\nClass Distribution:")
print(metadata_df['has_autism'].value_counts(normalize=True))


========== Fold 1/5 ==========

========== Fold 2/5 ==========

========== Fold 3/5 ==========

========== Fold 4/5 ==========

========== Fold 5/5 ==========


/tmp/ipython-input-235877396.py:99: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  self.data['encoded_label'] = self.label_encoder.fit_transform(self.data['has_autism'])
/tmp/ipython-input-235877396.py:99: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  self.data['encoded_label'] = self.label_encoder.fit_transform(self.data['has_autism'])


Downloading: "https://download.pytorch.org/models/convnext_tiny-983f1562.pth" to /root/.cache/torch/hub/checkpoints/convnext_tiny-983f1562.pth


100%|██████████| 109M/109M [00:00<00:00, 127MB/s]


Starting Training...
Epoch [1/30], Loss: 0.1304
Epoch [2/30], Loss: 0.1086
Epoch [3/30], Loss: 0.0913
Epoch [4/30], Loss: 0.0759


In [ ]:
import numpy as np # Import numpy
import matplotlib.pyplot as plt
import seaborn as sns

TP, FP, FN, TN = confusion_counts(y_true, y_pred)
# Create a 2x2 confusion matrix from these values
cm = np.array([[TN.item(), FP.item()],
               [FN.item(), TP.item()]]) # .item() to convert 0-dim tensor to scalar

plt.figure(figsize=(5, 4))

sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=["Pred No ASD", "Pred ASD"],
    yticklabels=["True No ASD", "True ASD"]
)

plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.title("Confusion Matrix – ASD Handwriting Screening")
plt.tight_layout()
plt.show()
import numpy as np

def bootstrap_ci(y_true, y_pred, metric_fn, n_boot=1000):
    scores = []
    n = len(y_true)
    for _ in range(n_boot):
        idx = np.random.choice(n, n, replace=True)
        scores.append(metric_fn(y_true[idx], y_pred[idx]))
    return np.percentile(scores, [2.5, 97.5])

ci_f1 = bootstrap_ci(y_true, y_pred, f1_score)
print("F1 95% CI:", ci_f1)

